# Delhi next-hour peak-demand classification

Big Data Essentials assignment. This standalone notebook contains all executable code; no GitHub clone or companion Python file is needed.

**Kaggle:** choose Add Input and attach [yug201/delhi-5-minute-electricity-demand-for-forecasting](https://www.kaggle.com/datasets/yug201/delhi-5-minute-electricity-demand-for-forecasting), then click Run All. The mandi notebook uses the historical 2022–2024 files, not a single-day snapshot. Kaggle Internet is only needed if a dependency is missing. The models run on CPU, including in T4 sessions.

**Colab:** open the implementation URL below and Run All. If `/kaggle/input` is absent, public data is downloaded through kagglehub; the mandi download is approximately 1.51 GB and can take substantially longer than computation.

No results are fabricated. All result and interpretation sections are generated from the input during execution.

## 1. Problem statement

Predict whether demand will exceed a training-derived peak threshold at any five-minute observation in the next hour. Practical application: advance warnings for Delhi grid planning.

## 2. Model and justification

Logistic Regression is a fast, interpretable main model. Compare demand/calendar-only and weather-enhanced versions against a shallow Decision Tree, persistence, yesterday and last-week baselines. Chronological splits and one-hour embargoes avoid overlapping target windows. Report sensitivity, specificity, precision, F1, ROC-AUC and average precision, including advance-warning-only observations.

## Configuration

Defaults work with the specified Kaggle inputs. Set `input_path` only when several compatible datasets are attached. Change `implementation_url` to your saved Kaggle notebook URL before the final submission. Timing is measured by stage and exported.

In [ ]:
TASK = 'electricity'
DATASET_SLUG = 'yug201/delhi-5-minute-electricity-demand-for-forecasting'
CONFIG = {'input_root': '/kaggle/input', 'input_path': None, 'output_dir': '/kaggle/working/electricity_results', 'implementation_url': 'https://colab.research.google.com/github/dhruvvvgg/bde-assignment/blob/main/notebooks/01_delhi_peak_classification.ipynb', 'peak_quantile': 0.9, 'bootstrap_repeats': 100, 'rolling_backtests': True}


## 3. Coding and implementation

Prepare dependencies, define helpers and execute the complete pipeline. Missing input causes an explicit setup error; the pipeline never substitutes synthetic data.

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

needed = ['holidays'] if TASK == 'electricity' else ['pyarrow']
for package in needed:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
if not Path(CONFIG['input_root']).exists() and CONFIG['input_path'] is None:
    if importlib.util.find_spec('kagglehub') is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'])
    import kagglehub
    CONFIG['input_root'] = kagglehub.dataset_download(DATASET_SLUG)
if not Path('/kaggle/working').exists() and CONFIG['output_dir'].startswith('/kaggle/working'):
    CONFIG['output_dir'] = str(Path.cwd() / (TASK + '_results'))
print('Device: CPU. A T4 is compatible but these models do not use GPU acceleration.')
print('Input:', CONFIG['input_root'])


In [ ]:
import json
import re
import time
import platform
import html
from pathlib import Path
from contextlib import contextmanager

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from threadpoolctl import threadpool_limits
import sklearn


def norm(value):
    return re.sub(r'[^a-z0-9]', '', str(value).casefold())


def find_column(columns, aliases, required=True):
    lookup = {norm(c): c for c in columns}
    found = next((lookup[norm(a)] for a in aliases if norm(a) in lookup), None)
    if found is None and required:
        raise ValueError(f'Missing column: {aliases}. Available columns: {list(columns)}')
    return found


def number(s):
    return pd.to_numeric(s.astype(str).str.replace(',', '', regex=False), errors='coerce').replace([np.inf, -np.inf], np.nan)


def table(df, name, out):
    df.to_csv(out / f'{name}.csv', index=False)
    display(df)
    return df


def figure(name, out):
    plt.tight_layout()
    plt.savefig(out / f'{name}.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()


def write_json(path, data):
    def clean(x):
        if isinstance(x, dict):
            return {str(k): clean(v) for k, v in x.items()}
        if isinstance(x, (list, tuple)):
            return [clean(v) for v in x]
        if isinstance(x, (np.integer, np.bool_)):
            return x.item()
        if isinstance(x, (float, np.floating)):
            return float(x) if np.isfinite(x) else None
        if isinstance(x, (pd.Timestamp, Path)):
            return str(x)
        return x
    path.write_text(json.dumps(clean(data), indent=2), encoding='utf-8')


@contextmanager
def timed(name, timings):
    start = time.perf_counter()
    yield
    timings[name] = round(time.perf_counter() - start, 3)


def discover(cfg, task):
    root = Path(cfg['input_root'])
    explicit = cfg.get('input_path')
    paths = [Path(explicit)] if explicit else sorted(root.rglob('*'))
    paths = [p for p in paths if p.is_file() and p.suffix.lower() in ('.csv', '.parquet')]
    aliases = ['power demand', 'Delhi demand', 'demand', 'load'] if task == 'electricity' else ['modal price', 'modalprice']
    accepted = []
    for p in paths:
        try:
            if p.suffix.lower() == '.csv':
                cols = pd.read_csv(p, nrows=0).columns
            else:
                import pyarrow.parquet as pq
                cols = pq.ParquetFile(p).schema.names
            if find_column(cols, aliases, False):
                accepted.append(p)
        except (OSError, ValueError) as exc:
            print(f'Skipped unreadable file {p.name}: {exc}')
    if not accepted:
        slug = 'yug201/delhi-5-minute-electricity-demand-for-forecasting' if task == 'electricity' else 'khandelwalmanas/daily-commodity-prices-india'
        raise FileNotFoundError(f'Attach https://www.kaggle.com/datasets/{slug} with Add Input, or set input_path/input_root. No compatible files found under {root}.')
    if task == 'electricity' and len(accepted) != 1:
        raise ValueError('Multiple demand files found. Set input_path to the intended Delhi CSV.')
    if task == 'mandi':
        stems = {p.stem for p in accepted if p.suffix.lower() == '.parquet'}
        accepted = [p for p in accepted if p.suffix.lower() != '.csv' or p.stem not in stems]
        if not explicit:
            lower, upper = pd.Timestamp(cfg['start_date']).year, pd.Timestamp(cfg['end_date']).year
            accepted = [p for p in accepted if not re.fullmatch(r'20\d{2}', p.stem) or lower <= int(p.stem) <= upper]
    if not accepted:
        raise FileNotFoundError('No files overlap the configured date window.')
    return accepted


def finish_report(out, cfg, sections, tables, timings, total_start):
    timings['total_seconds'] = round(time.perf_counter() - total_start, 3)
    table(pd.DataFrame([{'stage': k, 'seconds': v} for k, v in timings.items()]), 'runtime', out)
    write_json(out / 'run_config.json', cfg)
    write_json(out / 'environment.json', {'python': platform.python_version(), 'pandas': pd.__version__, 'numpy': np.__version__, 'sklearn': sklearn.__version__, 'device': 'CPU', 'timings': timings})
    body = '\n\n'.join(sections)
    (out / 'assignment_report.md').write_text(body, encoding='utf-8')
    markup = '<html><head><meta charset="utf-8"><title>BDE assignment results</title></head><body>'
    markup += '<pre style="white-space:pre-wrap">' + html.escape(body) + '</pre>'
    for name, df in tables.items():
        markup += '<h2>' + html.escape(name) + '</h2>' + df.to_html(index=False)
    for p in sorted(out.glob('*.png')):
        markup += f'<h2>{html.escape(p.stem)}</h2><img style="max-width:100%" src="{p.name}">'
    (out / 'assignment_report.html').write_text(markup + '</body></html>', encoding='utf-8')
    display(Markdown(body))
    import shutil
    archive = shutil.make_archive(str(out.parent / out.name), 'zip', out)
    print(f'Output folder: {out}\nReport bundle: {archive}\nImplementation: {cfg["implementation_url"]}')


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, accuracy_score, ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay
import holidays


def binary_metrics(y, pred, score=None):
    y, pred = np.asarray(y), np.asarray(pred)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if tp + fn else np.nan
    specificity = tn / (tn + fp) if tn + fp else np.nan
    return {'rows': len(y), 'peak_fraction': float(y.mean()) if len(y) else np.nan,
            'accuracy': accuracy_score(y, pred) if len(y) else np.nan,
            'precision': precision_score(y, pred, zero_division=0), 'recall_sensitivity': sensitivity,
            'specificity': specificity, 'f1': f1_score(y, pred, zero_division=0),
            'balanced_accuracy': (sensitivity + specificity) / 2,
            'roc_auc': roc_auc_score(y, score) if score is not None and len(np.unique(y)) == 2 else np.nan,
            'average_precision': average_precision_score(y, score) if score is not None and len(np.unique(y)) == 2 else np.nan,
            'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp}


def fit_cutoff(y, score):
    choices = np.linspace(.05, .95, 91)
    scores = [f1_score(y, score >= p, zero_division=0) for p in choices]
    return float(choices[np.argmax(scores)])


def electricity_frame(raw, cfg):
    dt = find_column(raw.columns, ['datetime', 'timestamp', 'date time'])
    power = find_column(raw.columns, ['power demand', 'Delhi demand', 'demand', 'load'])
    timestamps = pd.to_datetime(raw[dt], format='mixed', errors='coerce')
    if timestamps.dt.tz is not None:
        timestamps = timestamps.dt.tz_convert('Asia/Kolkata').dt.tz_localize(None)
    data = pd.DataFrame({'timestamp': timestamps, 'demand': number(raw[power])})
    weather = []
    for name, aliases in {'temperature': ['temp', 'temperature'], 'humidity': ['rhum', 'humidity', 'relative humidity'], 'dew_point': ['dwpt', 'dew point'], 'wind_speed': ['wspd', 'wind speed'], 'pressure': ['pres', 'pressure']}.items():
        source = find_column(raw.columns, aliases, False)
        if source:
            data[name] = number(raw[source])
            weather.append(name)
    if not {'temperature', 'humidity'}.issubset(weather):
        raise ValueError('The enhanced notebook expects the Delhi dataset with temperature and humidity columns.')
    invalid = data['timestamp'].isna() | data['demand'].isna() | data['demand'].le(0)
    quality = {'raw_rows': len(raw), 'invalid_demand_or_timestamp': int(invalid.sum())}
    data = data.loc[~invalid]
    quality['duplicate_timestamp_rows'] = int(data['timestamp'].duplicated().sum())
    quality['off_grid_timestamp_rows'] = int((data['timestamp'].dt.second.ne(0) | data['timestamp'].dt.minute.mod(5).ne(0)).sum())
    if quality['off_grid_timestamp_rows']:
        raise ValueError('Off-grid readings require explicit timestamp alignment; silent binning could change prediction timing.')
    grid = data.groupby('timestamp').mean(numeric_only=True).sort_index().resample('5min').asfreq()
    demand = grid['demand']
    quality.update({'grid_intervals': len(grid), 'missing_demand_intervals': int(demand.isna().sum()), 'start': str(grid.index.min()), 'end': str(grid.index.max())})
    frame = pd.DataFrame({'demand_now': demand})
    for minutes in [5, 15, 30, 60, 1440, 10080]:
        frame[f'lag_{minutes}min'] = demand.shift(minutes // 5)
    for minutes in [30, 60, 180]:
        win = demand.rolling(minutes // 5, min_periods=minutes // 5)
        frame[f'mean_{minutes}min'] = win.mean()
        frame[f'std_{minutes}min'] = win.std()
    frame['change_30min'] = demand - demand.shift(6)
    hour = frame.index.hour + frame.index.minute / 60
    for name, values, period in [('hour', hour, 24), ('weekday', frame.index.dayofweek, 7), ('month', frame.index.month, 12)]:
        frame[name + '_sin'] = np.sin(2 * np.pi * values / period)
        frame[name + '_cos'] = np.cos(2 * np.pi * values / period)
    frame['weekend'] = (frame.index.dayofweek >= 5).astype(int)
    calendar = holidays.country_holidays('IN', years=sorted(set(frame.index.year)))
    frame['india_holiday'] = [int(d.date() in calendar) for d in frame.index]
    base_features = list(frame.columns)
    for name in weather:
        frame[name] = grid[name]
    frame['temperature_x_humidity'] = frame['temperature'] * frame['humidity'] / 100
    weather_features = weather + ['temperature_x_humidity']
    future = pd.concat([demand.shift(-i) for i in range(1, 13)], axis=1)
    frame['future_hour_max'] = future.max(axis=1).where(future.notna().all(axis=1))
    frame['yesterday_hour_max'] = frame['future_hour_max'].shift(288)
    frame['last_week_hour_max'] = frame['future_hour_max'].shift(2016)
    keep = base_features + ['future_hour_max', 'yesterday_hour_max', 'last_week_hour_max']
    frame = frame.replace([np.inf, -np.inf], np.nan).dropna(subset=keep)
    quality['usable_examples'] = len(frame)
    quality['removed_incomplete_examples'] = len(grid) - len(frame)
    if len(frame) < 5000:
        raise ValueError('Fewer than 5,000 complete examples. Attach the full Delhi history.')
    return grid, frame, base_features, weather_features, quality


def block_bootstrap(test, predictions, repeats=100):
    days = test.index.normalize().unique()
    if len(days) < 8:
        return pd.DataFrame()
    rng = np.random.default_rng(42)
    y = test['target'].to_numpy()
    p = np.asarray(predictions)
    groups = [np.flatnonzero(test.index.normalize() == day) for day in days]
    samples = []
    for _ in range(repeats):
        indices = np.concatenate([groups[j] for j in rng.integers(0, len(groups), len(groups))])
        tn, fp, fn, tp = confusion_matrix(y[indices], p[indices], labels=[0, 1]).ravel()
        samples.append([2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else np.nan, tp / (tp + fn) if tp + fn else np.nan, tn / (tn + fp) if tn + fp else np.nan])
    vals = np.asarray(samples)
    return pd.DataFrame([{'metric': name, 'lower_95': np.nanquantile(vals[:, i], .025), 'upper_95': np.nanquantile(vals[:, i], .975), 'resampling_unit': 'calendar day'} for i, name in enumerate(['f1', 'recall_sensitivity', 'specificity'])])


def rolling_backtests(frame, base, weather, cfg):
    rows = []
    last = frame.index.max().normalize()
    for months_back in [12, 6, 0]:
        end = last - pd.DateOffset(months=months_back)
        eval_start = end - pd.Timedelta(days=90)
        tune_start = eval_start - pd.Timedelta(days=30)
        fit = frame.loc[frame.index + pd.Timedelta(hours=1) < tune_start].copy()
        tune = frame.loc[(frame.index >= tune_start) & (frame.index + pd.Timedelta(hours=1) < eval_start)].copy()
        check = frame.loc[(frame.index >= eval_start) & (frame.index + pd.Timedelta(hours=1) <= end)].copy()
        if min(len(fit), len(tune), len(check)) < 500:
            continue
        peak = fit.demand_now.quantile(cfg['peak_quantile'])
        for part in [fit, tune, check]:
            part['target'] = part.future_hour_max.ge(peak).astype(int)
        if fit.target.nunique() < 2 or tune.target.nunique() < 2:
            rows.append({'evaluation_start': eval_start, 'evaluation_end': end, 'status': 'skipped: one-class fit/tuning period'})
            continue
        columns = base + weather
        model = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=1500, random_state=42))
        with threadpool_limits(limits=4):
            model.fit(fit[columns], fit.target)
            cutoff = fit_cutoff(tune.target, model.predict_proba(tune[columns])[:, 1])
            score = model.predict_proba(check[columns])[:, 1]
        rows.append({'evaluation_start': eval_start, 'evaluation_end': end, 'status': 'evaluated', 'threshold_native_units': peak, 'probability_cutoff': cutoff, 'training_rows': len(fit), **binary_metrics(check.target, score >= cutoff, score)})
    return pd.DataFrame(rows)


def run_electricity(cfg):
    start = time.perf_counter()
    out = Path(cfg['output_dir'])
    out.mkdir(parents=True, exist_ok=True)
    timings = {}
    with timed('read_and_features', timings):
        path = discover(cfg, 'electricity')[0]
        raw = pd.read_parquet(path) if path.suffix == '.parquet' else pd.read_csv(path)
        grid, frame, base, weather, quality = electricity_frame(raw, cfg)
        quality['input_file'] = str(path)
    quality_table = table(pd.DataFrame([{'check': k, 'value': v} for k, v in quality.items()]), 'data_quality', out)
    i, j = int(len(frame) * .70), int(len(frame) * .85)
    vstart, tstart = frame.index[i], frame.index[j]
    train = frame.loc[frame.index + pd.Timedelta(hours=1) < vstart].copy()
    valid = frame.loc[(frame.index >= vstart) & (frame.index + pd.Timedelta(hours=1) < tstart)].copy()
    test = frame.loc[frame.index >= tstart].copy()
    threshold = float(train['demand_now'].quantile(cfg['peak_quantile']))
    for part in [train, valid, test]:
        part['target'] = (part['future_hour_max'] >= threshold).astype(int)
    if train['target'].nunique() < 2 or valid['target'].nunique() < 2:
        raise ValueError('Training or validation has only one class. Adjust peak_quantile or provide longer history.')
    assert train.index.max() + pd.Timedelta(hours=1) < valid.index.min()
    assert valid.index.max() + pd.Timedelta(hours=1) < test.index.min()
    splits = table(pd.DataFrame([{'split': name, 'rows': len(p), 'start': p.index.min(), 'end': p.index.max(), 'peak_fraction': p.target.mean()} for name, p in [('train', train), ('validation', valid), ('test', test)]]), 'split_summary', out)
    features = base + weather
    q1, q3 = train.demand_now.quantile([.25, .75])
    low, high = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    flags = grid[['demand']].copy()
    flags['outside_training_IQR_fences'] = flags.demand.lt(low) | flags.demand.gt(high)
    flags['absolute_change_5min_fraction'] = flags.demand.pct_change(fill_method=None).abs()
    unusual = flags.loc[flags.outside_training_IQR_fences | flags.absolute_change_5min_fraction.gt(.30)]
    unusual.to_csv(out / 'unusual_demand_for_review.csv', index_label='timestamp')
    table(pd.DataFrame([{'training_IQR_lower': low, 'training_IQR_upper': high, 'flagged_grid_rows': len(unusual), 'action': 'flag only; retain genuine peaks'}]), 'demand_outlier_audit', out)
    variants = {'Logistic Regression: demand/calendar': (base, make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=1500, random_state=42))),
                'Logistic Regression: weather enhanced': (features, make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=1500, random_state=42))),
                'Shallow Decision Tree': (features, make_pipeline(SimpleImputer(strategy='median'), DecisionTreeClassifier(max_depth=5, min_samples_leaf=max(50, len(train) // 1000), class_weight='balanced', random_state=42)))}
    fitted, rows, predictions, validation_rows = {}, [], {}, []
    with timed('train_and_evaluate_models', timings), threadpool_limits(limits=4):
        for name, (cols, model) in variants.items():
            tick = time.perf_counter()
            model.fit(train[cols], train.target)
            pv = model.predict_proba(valid[cols])[:, 1]
            cutoff = fit_cutoff(valid.target, pv)
            pt = model.predict_proba(test[cols])[:, 1]
            pred = (pt >= cutoff).astype(int)
            fitted[name] = (model, cols, cutoff)
            predictions[name] = (pred, pt)
            validation_rows.append({'model': name, 'probability_cutoff': cutoff, **binary_metrics(valid.target, pv >= cutoff, pv)})
            rows.append({'model': name, 'fit_and_score_seconds': time.perf_counter() - tick, **binary_metrics(test.target, pred, pt)})
    for name, source in [('Persistence', 'demand_now'), ('Same hour yesterday', 'yesterday_hour_max'), ('Same hour last week', 'last_week_hour_max')]:
        pred = (test[source] >= threshold).astype(int).to_numpy()
        predictions[name] = (pred, test[source].to_numpy())
        rows.append({'model': name, 'fit_and_score_seconds': 0, **binary_metrics(test.target, pred, test[source])})
    comparison = table(pd.DataFrame(rows), 'model_comparison_test', out)
    validation_results = table(pd.DataFrame(validation_rows), 'model_comparison_validation', out)
    main = 'Logistic Regression: weather enhanced'
    model, cols, cutoff = fitted[main]
    pred, prob = predictions[main]
    advance = test['demand_now'].lt(threshold).to_numpy()
    advance_table = table(pd.DataFrame([{'model': name, **binary_metrics(test.target.to_numpy()[advance], p[advance], s[advance])} for name, (p, s) in predictions.items()]), 'advance_warning_only', out)
    period_rows = []
    for period, ids in test.groupby(test.index.to_period('Q')).groups.items():
        mask = test.index.isin(ids)
        for name, (p, s) in predictions.items():
            period_rows.append({'period': str(period), 'model': name, **binary_metrics(test.target.to_numpy()[mask], p[mask], s[mask])})
    periods = table(pd.DataFrame(period_rows), 'quarterly_test_performance', out)
    with timed('historical_rolling_backtests', timings):
        backtests = table(rolling_backtests(train, base, weather, cfg) if cfg.get('rolling_backtests', True) else pd.DataFrame(), 'historical_rolling_backtests', out)
    thresholds = table(pd.DataFrame([{'probability_cutoff': c, **binary_metrics(test.target, prob >= c, prob)} for c in [.2, .35, .5, cutoff, .8]]), 'what_if_alert_cutoffs', out)
    uncertainty = table(block_bootstrap(test, pred, cfg['bootstrap_repeats']), 'day_block_confidence_intervals', out)
    coefficients = table(pd.DataFrame({'feature': cols, 'standardized_coefficient': model[-1].coef_[0]}).sort_values('standardized_coefficient'), 'coefficients', out)
    policy_rows = []
    for q in [.85, .90, .95]:
        policy = train.demand_now.quantile(q)
        policy_rows.append({'train_quantile': q, 'demand_threshold_native_units': policy, 'test_future_peak_fraction': test.future_hour_max.ge(policy).mean(), 'model_retrained': False})
    policy_table = table(pd.DataFrame(policy_rows), 'what_if_peak_definition', out)
    scenarios = []
    anchor = test.loc[test.demand_now.lt(threshold)].tail(1)[cols].copy()
    if not anchor.empty:
        for delta in [-3, 0, 3]:
            x = anchor.copy()
            x['temperature'] += delta
            if 'dew_point' in x:
                x['dew_point'] += delta
            x['temperature_x_humidity'] = x.temperature * x.humidity / 100
            scenarios.append({'temperature_change_C': delta, 'model_alert_score': model.predict_proba(x)[:, 1][0], 'interpretation': 'conditional model sensitivity; not a causal effect or calibrated probability'})
    scenario_table = table(pd.DataFrame(scenarios), 'what_if_temperature', out)
    export = test[['demand_now', 'future_hour_max', 'target']].copy()
    export['alert_score'], export['prediction'] = prob, pred
    export.to_csv(out / 'test_predictions.csv', index_label='timestamp')
    with timed('charts', timings):
        grid.demand.resample('D').max().plot(figsize=(11, 4), title='Delhi daily maximum demand; native source units')
        plt.axhline(threshold, color='red', linestyle='--', label='Training-derived peak threshold')
        plt.legend()
        figure('demand_history', out)
        ConfusionMatrixDisplay.from_predictions(test.target, pred, display_labels=['No peak', 'Peak'], colorbar=False)
        figure('confusion_matrix', out)
        if test.target.nunique() == 2:
            fig, axes = plt.subplots(1, 2, figsize=(11, 4))
            RocCurveDisplay.from_predictions(test.target, prob, ax=axes[0])
            PrecisionRecallDisplay.from_predictions(test.target, prob, ax=axes[1])
            axes[1].axhline(test.target.mean(), linestyle='--', color='grey', label='Peak prevalence')
            axes[1].legend()
            figure('roc_and_precision_recall', out)
        comparison.set_index('model')[['f1', 'recall_sensitivity', 'specificity']].plot.bar(figsize=(11, 4), ylim=(0, 1), rot=20)
        figure('model_comparison', out)
        coefficients.set_index('feature').standardized_coefficient.plot.barh(figsize=(9, 8), title='Standardized coefficients: associations, not causes')
        figure('coefficients', out)
        thresholds.plot(x='probability_cutoff', y=['precision', 'recall_sensitivity', 'specificity'], marker='o', figsize=(9, 4))
        figure('alert_cutoff_tradeoff', out)
    joblib.dump({'model': model, 'feature_columns': cols, 'alert_cutoff': cutoff, 'peak_threshold': threshold, 'config': cfg}, out / 'classifier.joblib')
    full = comparison.loc[comparison.model.eq(main)].iloc[0]
    persistence = comparison.loc[comparison.model.eq('Persistence')].iloc[0]
    no_weather = comparison.loc[comparison.model.eq('Logistic Regression: demand/calendar')].iloc[0]
    advance_main = advance_table.loc[advance_table.model.eq(main)].iloc[0]
    sections = ['# Classification: next-hour peak-demand warning in Delhi',
                f'## 1. Problem statement\nAt each observed five-minute timestamp, predict whether any demand reading in the following hour will exceed a threshold fixed from the training-period {cfg["peak_quantile"]:.0%} quantile. The application is an early warning for grid planning; the percentile is an experimental proxy, not an operator capacity limit.',
                '## 2. Model and justification\nLogistic Regression provides a fast, interpretable baseline for numeric demand, weather and calendar features. Standardization and regularization limit scale effects; balanced class weights address unequal class frequencies. A shallow Decision Tree tests nonlinear interactions without deep learning. Validation chooses alert cutoffs; the test period is untouched during fitting and selection. Class-weighted outputs are alert scores, not established calibrated probabilities.',
                f'## 3. Coding\nThe notebook contains the full pipeline. Current and historical features only; one-hour embargo at split boundaries. Missing demand is not interpolated, and incomplete feature/target windows are excluded. Weather imputation and scaling are learned only from training data. Training-derived IQR fences and sharp five-minute jumps flag unusual demand for review; genuine peaks are retained. Input: {path.name}.',
                f'## 4. Results\nRaw records: {len(raw):,}; usable examples: {len(frame):,}; missing demand slots: {quality["missing_demand_intervals"]:,}. Peak threshold: {threshold:.2f} native source units. Enhanced Logistic Regression test F1: {full.f1:.3f}; sensitivity: {full.recall_sensitivity:.3f}; specificity: {full.specificity:.3f}. Tables, confusion matrix, ROC/PR curves, quarterly results and day-block uncertainty are exported. Specificity = TN/(TN+FP), sensitivity = TP/(TP+FN).',
                f'## 5. Inference and what-if analysis\nEnhanced-model F1 minus persistence F1: {full.f1 - persistence.f1:+.3f}. Weather-model F1 minus demand/calendar model F1: {full.f1 - no_weather.f1:+.3f}; a negative difference means the added weather features did not help on this holdout. On timestamps currently below the peak threshold, sensitivity is {advance_main.recall_sensitivity:.3f}. This subset measures advance warning. Lower alert cutoffs trade false alarms against missed peaks; higher demand thresholds change event prevalence, and the policy table does not retrain models. Temperature scenarios show conditional score sensitivity, not causal effects. Three historical rolling backtests use expanding training, a separate 30-day tuning period and 90-day evaluation windows within the training era; these do not choose the final model. Quarterly results expose seasonal weakness; one chronological holdout and approximate day-block intervals do not prove deployment reliability. Demand units and weather timestamp availability must be confirmed before operational use.',
                f'## 6. URL of implementation\n{cfg["implementation_url"]}\nThis Colab URL opens the GitHub notebook. For a Kaggle submission, replace implementation_url with the saved Kaggle notebook URL and run again.']
    finish_report(out, cfg, sections, {'Model comparison': comparison, 'Advance warning': advance_table, 'Quarterly performance': periods, 'Historical backtests': backtests, 'Data quality': quality_table}, timings, start)
    return {'quality': quality, 'comparison': comparison, 'advance': advance_table, 'timings': timings, 'threshold': threshold, 'splits': (train, valid, test), 'model': model}


## 4. Results

This block runs preprocessing, model comparisons, metrics, charts, stability/sensitivity checks and report exports. Displayed tables are also saved as CSV files.

In [ ]:
RESULTS = run_electricity(CONFIG)

## 5. Inference and what-if analysis

The generated report below interprets the actual computed results. What-if scenarios are descriptive model sensitivities, not causal claims. Read the limitations alongside the findings.

In [ ]:
display(Markdown((Path(CONFIG['output_dir']) / 'assignment_report.md').read_text()))

## 6. URL of implementation

The default link opens this notebook in Colab from GitHub. Save a Kaggle version and place its actual URL in CONFIG if required. Export the generated ZIP from the Kaggle Output pane for the document and presentation.

In [ ]:
display(Markdown('**Implementation:** [' + CONFIG['implementation_url'] + '](' + CONFIG['implementation_url'] + ')'))
from IPython.display import FileLink
display(FileLink(str(Path(CONFIG['output_dir']).parent / (Path(CONFIG['output_dir']).name + '.zip'))))

## Presentation guide

1. Explain the practical problem and model choice.
2. Demonstrate the configuration and Run All pipeline.
3. Show data-quality and dataset-volume tables.
4. Explain model/cluster comparison charts and the appropriate metrics.
5. Discuss the generated inference, what-if checks and limitations.
6. Include the actual notebook implementation URL.

## References

- Dataset: https://www.kaggle.com/datasets/yug201/delhi-5-minute-electricity-demand-for-forecasting
- Logistic Regression: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
- K-means: https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html
- Confusion matrix: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html
- Adjusted Rand Index: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html
- Mandi quality-check reference: https://github.com/MrVinamra/Mandi-Price-Cointegration-VECM
- Delhi forecasting reference: https://github.com/pyaf/load_forecasting